<center><h1>Neuro-Fuzzy Computing Final Project: Text Classification with Neural Networks</h1></center>

## In this notebook, we go through the code of our project with short descriptions and comments where needed. Further analysis can be found in our report

### All necessary imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import utils
import tensorflow as tf

# Our modular package
from nlp_classifier import (
    DataLoader,
    TextPreprocessor,
    build_cnn_model,
    build_multicnn_model,
    build_bilstm_model,
    evaluate_model,
    measure_inference_time,
    plot_training_history,
    plot_confusion,
)

# For transformers
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import torch
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support as score,
)

<br><br><br><br>

### Explore the dataset: average text length, unique words and class balance

In [ ]:
df = pd.read_csv('news-classification.csv')

word_counts = df['content'].apply(lambda x: len(x.split()))
print(f'Average number of words per text: {word_counts.mean():.1f}')
print(f'Median number of words per text: {word_counts.median():.1f}')

all_words = set()
for text in df['content'].values:
    all_words.update(text.split())
print(f'Number of unique words: {len(all_words)}')

for col in ['category_level_1', 'category_level_2']:
    counts = df[col].value_counts()
    print(f'\n{col}:')
    print(f'  Classes: {len(counts)}')
    print(f'  Std dev of class sizes: {counts.std():.1f}')
    print(f'  Min class size: {counts.min()}')
    print(f'  Max class size: {counts.max()}')

#### It seems the distribution of texts within Level 1 is quite high which signifies a potential imbalance of the dataset
#### Distribution of texts within Level 2 is low, so it seems this category is quite balanced

<br><br>

### Define constants based on our observations

In [ ]:
MAX_SEQUENCE_LENGTH = 600  # Max length of input sequences
MAX_NUM_WORDS = 300000     # Max number of words to tokenize

<br>

### DataLoader class
Loads the dataset, splits into train/test, and returns data along with unique label arrays.
The class lives in `nlp_classifier/data_loader.py` and is fully reusable for any CSV dataset.

In [ ]:
data_loader = DataLoader('news-classification.csv')
X_train, y1_train, y2_train, X_test, y1_test, y2_test, y1_label, y2_label = data_loader.load_data()

# Shuffle training data to prevent the network from seeing categories in order
indices = np.arange(len(X_train))
np.random.shuffle(indices)
X_train = X_train[indices]
y1_train = y1_train[indices]
y2_train = y2_train[indices]

<br>

### TextPreprocessor
The `TextPreprocessor` class (in `nlp_classifier/preprocessor.py`) cleans text by:
- Converting to lowercase
- Replacing hyphens/dashes with spaces
- Removing digits
- Expanding contractions (e.g. *we're* → *we are*) using a pre-compiled regex
- Removing possessives
- Removing NLTK English stop words
- Removing single-character tokens

We only fit the tokenizer on the training data so the test set simulates unseen real-world data.

In [ ]:
text_preprocessor = TextPreprocessor(
    max_num_words=MAX_NUM_WORDS,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
text_preprocessor.fit_on_texts(X_train)
text_preprocessor.plot_token_concentration(30)

X_train_preprocessed = text_preprocessor.preprocess_text_data(X_train)
X_test_preprocessed = text_preprocessor.preprocess_text_data(X_test)

vocab_size = text_preprocessor.get_vocab_size()
print(f'Vocabulary size: {vocab_size}')
print(f'Train shape: {X_train_preprocessed.shape}  Test shape: {X_test_preprocessed.shape}')

<br>

### Encode labels for both category levels

In [ ]:
encoder1 = LabelEncoder()
encoder2 = LabelEncoder()

encoder1.fit(y1_label)
encoder2.fit(y2_label)

y1_train_enc = encoder1.transform(y1_train)
y2_train_enc = encoder2.transform(y2_train)
y1_test_enc = encoder1.transform(y1_test)
y2_test_enc = encoder2.transform(y2_test)

num_classes1 = np.max(y1_train_enc) + 1
num_classes2 = np.max(y2_train_enc) + 1

y1_train_cat = utils.to_categorical(y1_train_enc, num_classes1)
y2_train_cat = utils.to_categorical(y2_train_enc, num_classes2)
y1_test_cat = utils.to_categorical(y1_test_enc, num_classes1)
y2_test_cat = utils.to_categorical(y2_test_enc, num_classes2)

print(f'Level 1 classes: {num_classes1}  Level 2 classes: {num_classes2}')

<br><br><br>

## LEVEL 1 (CLASS) CLASSIFICATION

<br>

### 1a. Original CNN Architecture
Single-kernel 1-D CNN with BatchNorm, LeakyReLU, GlobalMaxPooling, and Dropout.
Compiled with RMSprop (lr=0.0007, rho=0.5) and categorical cross-entropy.

In [ ]:
embedding_dim = 100
batch_size = 32
epochs = 25

level1_cnn = build_cnn_model(
    vocab_size=vocab_size,
    num_classes=num_classes1,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=embedding_dim,
)

X_train_new, X_val, y1_train_new, y1_val = train_test_split(
    X_train_preprocessed, y1_train_cat, test_size=0.1, random_state=42
)

level1_cnn.summary()

### Train the CNN

In [ ]:
start_time = time.time()
history_l1_cnn = level1_cnn.fit(
    X_train_new, y1_train_new,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y1_val),
)
print(f'\nTraining took: {time.time()-start_time:.2f} sec')

### Plot training curves

In [ ]:
plot_training_history(history_l1_cnn, 'Level 1 CNN')

### Evaluate on test set

In [ ]:
pred_l1_cnn, true_l1, acc_l1_cnn, *_ = evaluate_model(
    level1_cnn, X_test_preprocessed, y1_test_cat, encoder1, 'Level 1 CNN'
)
measure_inference_time(level1_cnn, X_test_preprocessed)
plot_confusion(true_l1, pred_l1_cnn, 'Level 1 CNN – Confusion Matrix')

<br><br>

### 1b. Multi-Kernel CNN Architecture
Three parallel Conv1D branches with kernel sizes (2, 3, 4) capture different n-gram features.
Outputs are concatenated after GlobalMaxPooling, then passed through dense layers.
Compiled with Adam optimizer.

In [ ]:
level1_multicnn = build_multicnn_model(
    vocab_size=vocab_size,
    num_classes=num_classes1,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=128,
    num_filters=256,
)
level1_multicnn.summary()

### Train the Multi-CNN

In [ ]:
start_time = time.time()

# Early stopping to prevent overfitting
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
# Learning rate reduction on plateau
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
)

history_l1_mcnn = level1_multicnn.fit(
    X_train_new, y1_train_new,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y1_val),
    callbacks=[early_stop, lr_scheduler],
)
print(f'\nTraining took: {time.time()-start_time:.2f} sec')

### Plot training curves

In [ ]:
plot_training_history(history_l1_mcnn, 'Level 1 Multi-CNN')

### Evaluate on test set

In [ ]:
pred_l1_mcnn, _, acc_l1_mcnn, *_ = evaluate_model(
    level1_multicnn, X_test_preprocessed, y1_test_cat, encoder1, 'Level 1 Multi-CNN'
)
measure_inference_time(level1_multicnn, X_test_preprocessed)
plot_confusion(true_l1, pred_l1_mcnn, 'Level 1 Multi-CNN – Confusion Matrix')

<br><br>

### 1c. Bidirectional LSTM Architecture
A BiLSTM captures long-range dependencies from both directions of the text.
Followed by BatchNorm, dropout, and dense layers. Compiled with Adam.

In [ ]:
level1_bilstm = build_bilstm_model(
    vocab_size=vocab_size,
    num_classes=num_classes1,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=128,
    lstm_units=128,
)
level1_bilstm.summary()

### Train the BiLSTM

In [ ]:
start_time = time.time()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
)

history_l1_lstm = level1_bilstm.fit(
    X_train_new, y1_train_new,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y1_val),
    callbacks=[early_stop, lr_scheduler],
)
print(f'\nTraining took: {time.time()-start_time:.2f} sec')

### Plot training curves

In [ ]:
plot_training_history(history_l1_lstm, 'Level 1 BiLSTM')

### Evaluate on test set

In [ ]:
pred_l1_lstm, _, acc_l1_lstm, *_ = evaluate_model(
    level1_bilstm, X_test_preprocessed, y1_test_cat, encoder1, 'Level 1 BiLSTM'
)
measure_inference_time(level1_bilstm, X_test_preprocessed)
plot_confusion(true_l1, pred_l1_lstm, 'Level 1 BiLSTM – Confusion Matrix')

<br><br>

### Level 1 – Model Comparison

In [ ]:
print('\nLevel 1 Accuracy Comparison')
print(f'  CNN (original):  {acc_l1_cnn:.4f}')
print(f'  Multi-CNN:       {acc_l1_mcnn:.4f}')
print(f'  BiLSTM:          {acc_l1_lstm:.4f}')

<br><br><br>

## LEVEL 2 (SUBCLASS) CLASSIFICATION

<br>

### 2a. Original CNN Architecture (Level 2)
Same architecture as Level 1 CNN, but with 109 output classes.

In [ ]:
level2_cnn = build_cnn_model(
    vocab_size=vocab_size,
    num_classes=num_classes2,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=embedding_dim,
)

X_train_new2, X_val2, y2_train_new, y2_val = train_test_split(
    X_train_preprocessed, y2_train_cat, test_size=0.1, random_state=42
)

### Train

In [ ]:
start_time = time.time()
history_l2_cnn = level2_cnn.fit(
    X_train_new2, y2_train_new,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val2, y2_val),
)
print(f'\nTraining took: {time.time()-start_time:.2f} sec')

### Plot and evaluate

In [ ]:
plot_training_history(history_l2_cnn, 'Level 2 CNN')
pred_l2_cnn, true_l2, acc_l2_cnn, *_ = evaluate_model(
    level2_cnn, X_test_preprocessed, y2_test_cat, encoder2, 'Level 2 CNN'
)
measure_inference_time(level2_cnn, X_test_preprocessed)

<br><br>

### 2b. Multi-Kernel CNN Architecture (Level 2)

In [ ]:
level2_multicnn = build_multicnn_model(
    vocab_size=vocab_size,
    num_classes=num_classes2,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=128,
    num_filters=256,
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
)

start_time = time.time()
history_l2_mcnn = level2_multicnn.fit(
    X_train_new2, y2_train_new,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val2, y2_val),
    callbacks=[early_stop, lr_scheduler],
)
print(f'\nTraining took: {time.time()-start_time:.2f} sec')

plot_training_history(history_l2_mcnn, 'Level 2 Multi-CNN')
pred_l2_mcnn, _, acc_l2_mcnn, *_ = evaluate_model(
    level2_multicnn, X_test_preprocessed, y2_test_cat, encoder2, 'Level 2 Multi-CNN'
)
measure_inference_time(level2_multicnn, X_test_preprocessed)

<br><br>

### 2c. Bidirectional LSTM (Level 2)

In [ ]:
level2_bilstm = build_bilstm_model(
    vocab_size=vocab_size,
    num_classes=num_classes2,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=128,
    lstm_units=128,
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
)

start_time = time.time()
history_l2_lstm = level2_bilstm.fit(
    X_train_new2, y2_train_new,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val2, y2_val),
    callbacks=[early_stop, lr_scheduler],
)
print(f'\nTraining took: {time.time()-start_time:.2f} sec')

plot_training_history(history_l2_lstm, 'Level 2 BiLSTM')
pred_l2_lstm, _, acc_l2_lstm, *_ = evaluate_model(
    level2_bilstm, X_test_preprocessed, y2_test_cat, encoder2, 'Level 2 BiLSTM'
)
measure_inference_time(level2_bilstm, X_test_preprocessed)

<br><br>

### Level 2 – Model Comparison

In [ ]:
print('\nLevel 2 Accuracy Comparison')
print(f'  CNN (original):  {acc_l2_cnn:.4f}')
print(f'  Multi-CNN:       {acc_l2_mcnn:.4f}')
print(f'  BiLSTM:          {acc_l2_lstm:.4f}')

<br><br><br><br><br><br><br><br><br>

# Transformers Implementation

### We use the DistilBERT model (`distilbert-base-uncased`) for transfer learning.

**Training parameters:**
- Learning rate: 2e-5
- Optimizer: AdamW
- Train batch size: 16
- Eval batch size: 64
- Warmup steps: 500
- Weight decay: 1e-5
- Epochs: 15

<br><br><br>

## Transformer – Level 1 Classification

### Preprocessing

In [ ]:
pretrained_model = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

def preprocess_function(txt):
    return tokenizer(txt['text'], truncation=True, padding=True, max_length=512)

def pipeline(dataframe):
    dataset = Dataset.from_pandas(dataframe, preserve_index=False)
    tokenized_ds = dataset.map(preprocess_function, batched=True)
    tokenized_ds = tokenized_ds.remove_columns('text')
    return tokenized_ds

df_t1 = pd.read_csv('news-classification.csv')
df_t1 = df_t1[['content', 'category_level_1']].rename(
    columns={'content': 'text', 'category_level_1': 'label'}
)

unique_labels = df_t1['label'].unique()
df_t1 = df_t1.sample(frac=1).reset_index(drop=True)
label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
df_t1['label'] = df_t1['label'].map(label_to_index)

train_df, val_df = train_test_split(df_t1[['text', 'label']], test_size=0.2, random_state=42)

tokenized_train = pipeline(train_df)
tokenized_val = pipeline(val_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Initialize and train

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model, num_labels=17
).to(device)

training_args = TrainingArguments(
    output_dir='./results',
    save_strategy='epoch',
    optim='adamw_torch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=15,
    warmup_steps=500,
    weight_decay=1e-5,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

### Save and evaluate

In [ ]:
trainer.save_model('./level1_model')
tokenizer.save_pretrained('./level1_model')

preds = trainer.predict(tokenized_val)
preds_flat = [np.argmax(x) for x in preds[0]]

cm = confusion_matrix(val_df['label'], preds_flat)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

precision, recall, fscore, support = score(val_df['label'], preds_flat)
accuracy = accuracy_score(val_df['label'], preds_flat)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F-score: {fscore}')

<br><br><br><br><br><br>

## Transformer – Level 2 Classification

### Preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

df_t2 = pd.read_csv('news-classification.csv')
df_t2 = df_t2[['content', 'category_level_2']].rename(
    columns={'content': 'text', 'category_level_2': 'label'}
)

unique_labels = df_t2['label'].unique()
df_t2 = df_t2.sample(frac=1).reset_index(drop=True)
label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
df_t2['label'] = df_t2['label'].map(label_to_index)

train_df2, val_df2 = train_test_split(df_t2[['text', 'label']], test_size=0.2, random_state=42)

tokenized_train2 = pipeline(train_df2)
tokenized_val2 = pipeline(val_df2)
data_collator2 = DataCollatorWithPadding(tokenizer=tokenizer)

### Initialize and train

In [ ]:
model2 = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model, num_labels=109
).to(device)

training_args2 = TrainingArguments(
    output_dir='./results2',
    save_strategy='epoch',
    optim='adamw_torch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=15,
    warmup_steps=500,
    weight_decay=1e-5,
    report_to='none',
)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=tokenized_train2,
    eval_dataset=tokenized_val2,
    tokenizer=tokenizer,
    data_collator=data_collator2,
)

trainer2.train()

### Save and evaluate

In [ ]:
trainer2.save_model('./saved_model2')
tokenizer.save_pretrained('./saved_model2')

preds2 = trainer2.predict(tokenized_val2)
preds_flat2 = [np.argmax(x) for x in preds2[0]]

precision, recall, fscore, support = score(val_df2['label'], preds_flat2)
accuracy = accuracy_score(val_df2['label'], preds_flat2)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F-score: {fscore}')

<br><br><br>

#### In both transformer models, we notice an increase in accuracy compared to our initial implementations,
#### showing that there is potential for more improvement with hyperparameter fine-tuning.